# Mamba from scratch — JAX + Equinox

**Runtime:** Colab with a T4 (`Runtime → Change runtime type → T4 GPU`). It also runs on CPU
if you shrink `Config` — see the note in §0.

## 0. Setup, data, and one config cell

*Plumbing. Read the config, skim the rest.*

Colab already ships a `jax` built against its CUDA plugin. **Do not upgrade `jax` here** — pip
would happily install a version that no longer matches the preinstalled GPU plugin, and you
would silently fall back to CPU.

In [ ]:
!pip install -q equinox optax tokenizers huggingface_hub

In [ ]:
import collections
import math
import re
import textwrap
import time
from dataclasses import dataclass

import numpy as np
import jax
import jax.numpy as jnp
import equinox as eqx
import optax
import matplotlib.pyplot as plt
from IPython.display import clear_output

print("jax", jax.__version__, "| equinox", eqx.__version__, "| devices:", jax.devices())

### The config

Every size in the notebook comes from here, so there is exactly one place to shrink things.

A word on `batch = 16` rather than 32. Our scan materialises the full state trajectory of shape
`(batch, seq_len, d_inner, d_state)` per layer — at these numbers about 134 MB per array — and
autodiff keeps it alive for the backward pass. The real Mamba never materialises it: its fused
CUDA kernel keeps the state in SRAM and *recomputes* it during the backward pass. This is the
concrete meaning of the lecture's claim that half of Mamba's contribution is the kernel. We are
writing the portable half, and we pay for it in memory.

**Running on CPU instead:** `Config(d_model=128, n_layer=4, seq_len=128, batch=8, steps=600,
warmup=50, eval_every=100, eval_batches=5)` takes about five minutes on a laptop and still gets to
recognisable stories (measured: validation loss 9.4 → 3.2, samples like *"Once upon a time, there
was a little girl named Sue."*).

In [ ]:
@dataclass(frozen=True)
class Config:
    # --- model ---
    d_model: int = 256      # width of the residual stream
    n_layer: int = 6
    expand: int = 2         # d_inner = expand * d_model — the SSM runs in a wider space
    d_state: int = 16       # N: size of the state per channel
    d_conv: int = 4         # taps of the causal depthwise convolution
    dt_rank: int = 16       # low-rank bottleneck for Δ (paper: ceil(d_model / 16))

    # --- data ---
    seq_len: int = 256
    batch: int = 16

    # --- optimisation ---
    steps: int = 3000
    lr: float = 3e-3
    lr_min: float = 1e-4
    warmup: int = 100
    weight_decay: float = 0.1
    grad_clip: float = 1.0
    eval_every: int = 250
    eval_batches: int = 20
    seed: int = 0

    @property
    def d_inner(self) -> int:
        return self.expand * self.d_model


cfg = Config()
cfg

### Data: TinyStories, tokenized with the GPT-2 BPE

[TinyStories](https://huggingface.co/datasets/roneneldan/TinyStories) is synthetic children's
stories written with a deliberately small vocabulary. That is exactly what we want: a model of a
few million parameters can actually learn the distribution, so after ten minutes of training the
samples are readable English rather than noise — and readable samples are what make the rest of
the seminar checkable by eye.

We take the **validation** file (22 MB, ≈5.5M tokens). The training file is 2.2 GB and we do not
need it: at our model size the validation split is already more data than we will consume.

For the tokenizer we take GPT-2's `tokenizer.json` directly. The `tokenizers` package loads it on
its own — `transformers` is not needed.

In [ ]:
from huggingface_hub import hf_hub_download
from tokenizers import Tokenizer

corpus_path = hf_hub_download(
    repo_id="roneneldan/TinyStories",
    filename="TinyStoriesV2-GPT4-valid.txt",
    repo_type="dataset",
)
tok = Tokenizer.from_file(hf_hub_download("openai-community/gpt2", "tokenizer.json"))

text = open(corpus_path, encoding="utf-8").read()
print(f"{len(text) / 1e6:.1f} MB of text\n")
print(text[:350].strip(), "\n...")

#### Encoding, and why we prune the vocabulary

GPT-2 has 50257 tokens. TinyStories uses only a fraction of them, and an embedding table of
`50257 × 256` would be 12.9M parameters — about five times the Mamba stack itself, most of it rows
that never receive a gradient.

So we keep only the ids that occur in the corpus and renumber them compactly. It is a handful of lines,
it costs nothing at inference (we map back before decoding), and it moves the parameter budget
from the vocabulary into the model, which is the part this seminar is about.

In [ ]:
SEP = "<|endoftext|>"
eot_gpt2 = tok.token_to_id(SEP)

docs = [d.strip() for d in text.split(SEP) if d.strip()]
ids = np.concatenate([np.asarray(e.ids + [eot_gpt2], dtype=np.int32)
                      for e in tok.encode_batch(docs)])
print(f"{len(docs):,} stories -> {len(ids):,} GPT-2 tokens")

# --- vocabulary pruning -------------------------------------------------------------------
keep_ids = np.unique(ids)                                 # GPT-2 ids TinyStories actually uses
old2new = np.full(tok.get_vocab_size(), -1, dtype=np.int32)
old2new[keep_ids] = np.arange(len(keep_ids), dtype=np.int32)

data = old2new[ids].astype(np.uint16)
vocab_size = len(keep_ids)
eot = int(old2new[eot_gpt2])


def encode(s: str, strict: bool = True) -> list:
    gpt2 = tok.encode(s).ids
    out = old2new[gpt2]
    if (out < 0).any():
        missing = [tok.decode([g]) for g, n in zip(gpt2, out) if n < 0]
        shown = missing[:6] + (["..."] if len(missing) > 6 else [])
        if strict:
            raise ValueError(f"TinyStories never uses {len(missing)} of these tokens: {shown}")
        print(f"  [dropped {len(missing)} token(s) outside the corpus: {shown}]")
    return [int(i) for i in out if i >= 0]


def decode(seq) -> str:
    return tok.decode([int(keep_ids[i]) for i in seq])


print(f"vocabulary: {tok.get_vocab_size():,} -> {vocab_size:,} "
      f"({vocab_size * cfg.d_model / 1e6:.1f}M embedding params instead of "
      f"{tok.get_vocab_size() * cfg.d_model / 1e6:.1f}M)")
print("round-trip:", repr(decode(data[:20])))

In [ ]:
# Held-out tail for validation. Sampling random windows out of one flat array is all the
# "data loader" a language model of this size needs.
n_val = len(data) // 10
train_data, val_data = data[:-n_val], data[-n_val:]


def get_batch(rng, split, batch=None, seq_len=None):
    src = train_data if split == "train" else val_data
    batch = batch or cfg.batch
    seq_len = seq_len or cfg.seq_len
    i = rng.integers(0, len(src) - seq_len - 1, size=batch)
    x = np.stack([src[j:j + seq_len] for j in i]).astype(np.int32)
    y = np.stack([src[j + 1:j + 1 + seq_len] for j in i]).astype(np.int32)
    return jnp.asarray(x), jnp.asarray(y)


rng = np.random.default_rng(cfg.seed)
xb, yb = get_batch(rng, "train")
print(f"train {len(train_data):,} tokens | val {len(val_data):,} tokens | batch {xb.shape}")

## 1. A sequence model that is a physical system — ≈10 min

A Transformer answers *"what should I remember?"* by remembering everything. The KV cache grows
with the sequence, and the cost of producing the next token grows with it. An RNN answers with a
fixed-size state — $O(1)$ per token — but a nonlinear RNN cannot be unrolled in parallel, so it
does not train at scale.

S4 starts somewhere else entirely: not from attention, but from a **linear dynamical system**
borrowed from control theory.

$$
h'(t) \;=\; A\,h(t) + B\,x(t),
\qquad
y(t) \;=\; C\,h(t) + D\,x(t)
$$

- $x(t) \in \mathbb{R}$ — one scalar channel of the input, viewed as a function of continuous time;
- $h(t) \in \mathbb{R}^{N}$ — the state: a fixed-size summary of everything $x$ has done so far;
- $A \in \mathbb{R}^{N \times N}$, $B \in \mathbb{R}^{N \times 1}$, $C \in \mathbb{R}^{1 \times N}$,
  $D \in \mathbb{R}$ — the system itself.

Three things to notice before we touch any code.

**Nothing here mentions tokens.** This is a claim about *compression*: that $N$ numbers can hold a
useful summary of an entire signal. Which $A$ makes that claim true is exactly what the S4 paper
contributes — the **HiPPO** matrix, for which $h(t)$ holds the coefficients of the projection of
the past of $x$ onto a polynomial basis, i.e. a provably good fixed-budget compression of history.
Mamba keeps only the diagonal shadow of that construction, so we will not need HiPPO itself; but
it is the reason anyone believed a fixed state could work in the first place.

**$D$ is a skip connection.** It routes $x$ past the state entirely. Keep it in mind — it survives
into the final architecture unchanged, as a per-channel residual.

**Linearity is the whole point.** Because the system is linear, the state at any time has a closed
form — no numerical integration required:

$$
h(t) \;=\; e^{A(t - t_0)}\,h(t_0) \;+\; \int_{t_0}^{t} e^{A(t-s)}\,B\,x(s)\,ds
$$

Read it as: *what the state was, decayed forward by $e^{A\cdot\text{elapsed}}$, plus every bit of
input the system has absorbed since, each decayed by how long ago it arrived.* This integral is
the only thing the next section needs.

**One system per channel.** A real layer has `d_inner` channels, and each gets its own copy of the
system and its own state. So `h` is `(d_inner, N)` and channels never mix inside the SSM — mixing
is the job of the linear projections around it. This is worth holding onto: it is why every
formula below is elementwise in the channel index.

## 2. Discretization: from an ODE to an update rule — ≈12 min

A differential equation is not something you can feed a token sequence. We need an update rule.

Pick a step size $\Delta$ — *"how much time one token takes"* — and ask where the state lands
after $\Delta$. Take the closed form from §1 with $t_0 = (t-1)\Delta$ and $t = t\Delta$, and make
the **zero-order hold** assumption: the input is constant over the interval, $x(s) = x_t$. Then
$x_t$ comes out of the integral:

$$
h_t \;=\; e^{\Delta A}\,h_{t-1} \;+\; \left(\int_0^{\Delta} e^{A\tau}\,d\tau\right) B\,x_t
\qquad (\tau = t\Delta - s)
$$

The remaining integral is elementary — $\int_0^{\Delta} e^{A\tau} d\tau = A^{-1}(e^{\Delta A} - I)$ —
and we are done:

$$
\boxed{\;
\bar A = \exp(\Delta A),
\qquad
\bar B = (\Delta A)^{-1}\!\left(e^{\Delta A} - I\right)\Delta B,
\qquad
h_t = \bar A h_{t-1} + \bar B x_t,
\quad
y_t = C h_t + D x_t
\;}
$$

The differential equation is gone. What is left is an **RNN update** — and, crucially, a *linear*
one, which is what will let us parallelise it in §5.

### Two readings you should carry to the end of the notebook

**(1) $\Delta$ is a knob on attention span.** As $\Delta \to 0$: $\bar A \to I$ and $\bar B \to 0$,
so the state is left untouched — *this token was ignored*. As $\Delta$ grows: $\bar A \to 0$ and
the old state is wiped, the state becomes a function of the current token — *focus on this one*.
Nobody designed this gate. It fell out of asking what a time step means.

**(2) Diagonal $A$ gives a per-channel forget gate.** Take $A = \mathrm{diag}(a_1,\dots,a_N)$ with
every $a_n < 0$. Then $\bar A = \mathrm{diag}(e^{\Delta a_1}, \dots, e^{\Delta a_N})$, each entry in
$(0, 1)$: every coordinate of the state decays at *its own* rate, with a memory horizon of about
$1/(\Delta|a_n|)$ steps. This is literally the per-channel decay of gated linear attention — derived
from physics rather than invented. The whole rest of the notebook uses diagonal $A$, so from here
on $\bar A$ is a vector and every operation on it is elementwise.

In [ ]:
# Reading (1), drawn: what Δ does to the retention factor Ā = e^{Δa}.
deltas = np.logspace(-3, 1, 300)
plt.figure(figsize=(5.6, 3.2))
for a in (-0.1, -1.0, -4.0, -16.0):
    plt.semilogx(deltas, np.exp(deltas * a), label=f"$a = {a}$")
plt.axhline(1.0, lw=0.6, c="k")
plt.axhline(0.0, lw=0.6, c="k")
plt.xlabel(r"$\Delta$   (how much time this token takes)")
plt.ylabel(r"$\bar A = e^{\Delta a}$")
plt.title("keep the state  ←→  overwrite it")
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

So the recurrence we will actually implement, with $A$ diagonal and everything elementwise per
channel, is

$$
h_t = \exp(\Delta A) \odot h_{t-1} \;+\; (\Delta B)\, x_t,
\qquad
y_t = C \cdot h_t + D\,x_t .
$$

Every symbol on the right is now something we can put in an array. The only question left is where
$\Delta, B, C$ come from — and that question is the difference between S4 and Mamba.

## 3. S4: the LTI case, and why it is one long convolution — ≈10 min

Suppose $\Delta, B, C$ are **parameters**, the same for every position. Then $\bar A, \bar B, C$ do
not depend on $t$, and the system is *linear time-invariant*. Unroll the recurrence from
$h_{-1} = 0$:

$$
h_t = \sum_{i \ge 0} \bar A^{\,i} \bar B\, x_{t-i}
\qquad\Longrightarrow\qquad
y_t = \sum_{i \ge 0} \underbrace{C \bar A^{\,i} \bar B}_{\bar K_i}\, x_{t-i}
$$

The output is the input convolved with a fixed kernel

$$
\bar K = \bigl(C\bar B,\; C\bar A\bar B,\; C\bar A^{2}\bar B,\; \dots\bigr) \in \mathbb{R}^{L}
\qquad\text{i.e.}\qquad y = x * \bar K .
$$

That is S4's answer to "how do you train an RNN in parallel": **you don't** — you notice it is a
convolution, and evaluate it with an FFT in $O(L\log L)$, all positions at once. Recurrent form for
inference, convolutional form for training, same parameters.

Let us check that these really are the same computation.

In [ ]:
def lti_kernel(A_bar, B_bar, C, L):
    """K_i = C A_bar^i B_bar for i = 0..L-1, with a diagonal A_bar given as a vector."""
    powers = A_bar[None, :] ** np.arange(L)[:, None]      # (L, N)
    return (powers * (C * B_bar)[None, :]).sum(-1)        # (L,)


g = np.random.default_rng(0)
N, L = 8, 64
A_bar = np.exp(-np.exp(g.normal(size=N)))                 # e^{Δa} for some Δ, a<0  ->  (0, 1)
B_bar, C = g.normal(size=N), g.normal(size=N)
x = g.normal(size=L)

# (a) the recurrence, one step at a time
h, y_rec = np.zeros(N), []
for t in range(L):
    h = A_bar * h + B_bar * x[t]
    y_rec.append(C @ h)
y_rec = np.array(y_rec)

# (b) one long convolution, evaluated by FFT
K = lti_kernel(A_bar, B_bar, C, L)
n = 1 << (2 * L - 1).bit_length()
y_fft = np.fft.irfft(np.fft.rfft(K, n) * np.fft.rfft(x, n), n)[:L]

print("max |recurrence - FFT convolution| =", np.abs(y_rec - y_fft).max())

Same numbers, two very different cost profiles. But look at what we had to assume to get here:
$\bar A, \bar B, C$ **fixed for every position**. An LTI system does the same thing to every token
regardless of what the token is. It has no way to say *"this one matters, that one is filler"* —
its decay schedule is fixed before it sees any input.

That is precisely the property that has to break.

## 4. S6: selectivity, and every parametrization choice — ≈13 min

The Mamba paper makes the point with two toy tasks that an LTI model provably cannot do:

- **selective copying** — a few content tokens scattered among filler; reproduce them in order.
  Requires *skipping* by content, and content-blind decay cannot skip.
- **induction heads** — having seen `... A B ...`, on a later `A` predict `B`. Requires storing a
  particular earlier token because of what it was.

Both need the update to depend on *what the token is*. So: make it.

$$
\Delta_t = f_\Delta(x_t), \qquad B_t = f_B(x_t), \qquad C_t = f_C(x_t)
$$

$A$ stays a plain parameter. It does not need to be selective: it is the *basis of timescales*, and
$\Delta_t$ already modulates it multiplicatively inside $\exp(\Delta_t A)$ — that product is where
selectivity enters. Making $A$ input-dependent as well would buy nothing and cost a matrix per token.

**The price is immediate.** $\bar A_t = \exp(\Delta_t A)$ now changes at every step, so there is no
fixed kernel $\bar K$, and §3's convolution is gone. §5 is about getting the parallelism back.

### The parametrization, choice by choice

| what | how | why |
|---|---|---|
| $A$ | $A = -\exp(A_{\log})$, stored as $A_{\log}$, initialised $\log n$ for $n = 1..N$ | the exponential *forces* $A<0$, so $e^{\Delta a} \in (0,1)$ and the state can never blow up, for any value the optimiser wanders to. The init ($a_n = -n$, "S4D-Real") gives the $N$ channels a spread of decay rates instead of $N$ copies of the same one. |
| $\Delta_t$ | $\Delta_t = \mathrm{softplus}\bigl(b_\Delta + W_{\uparrow} W_{\downarrow} x_t\bigr)$ | `softplus` keeps $\Delta > 0$ (a negative time step is meaningless) and is smooth at 0, unlike ReLU. The projection is **low-rank** through `dt_rank ≈ d_model/16`: a full `d_inner × d_inner` map would dominate the parameter count for a scalar-per-channel quantity. |
| $b_\Delta$ | $b_\Delta = \mathrm{softplus}^{-1}(\delta)$, $\;\delta \sim \mathrm{LogUniform}(0.001,\,0.1)$ | this is the init that matters most. It puts the channels' *initial* memory horizons $\approx 1/\Delta$ across three orders of magnitude, so some channels start as near-copies of the input and others as long-run integrators, and the optimiser can pick. |
| $B_t, C_t$ | one projection $x_t \mapsto (\text{dt\_rank} + 2N)$, split three ways | $B_t$ decides *how strongly this token is written into the state*, $C_t$ decides *what is read out of it at this position*. Compare with attention: $B$ is a key-like write, $C$ a query-like read — the resemblance is not a coincidence, and Mamba-2 makes it an identity. |
| $D$ | per-channel scalar, initialised to 1 | the skip from §1. Lets a channel pass the token straight through without going near the state. |

### The final formulas

For one channel, at position $t$, with everything below elementwise over `(d_inner, N)`:

$$
\boxed{\;
\bar A_t = \exp\!\left(\Delta_t A\right),
\qquad
h_t = \bar A_t \odot h_{t-1} + \left(\Delta_t B_t\right) x_t,
\qquad
y_t = C_t \cdot h_t + D\,x_t
\;}
$$

Shapes, which is how the code will read: $\Delta_t$ is `(d_inner,)`, $A$ is `(d_inner, N)`,
$B_t, C_t$ are `(N,)`, $h_t$ is `(d_inner, N)`, $x_t$ and $y_t$ are `(d_inner,)`.

In [ ]:
# The Δ initialisation, made concrete: what memory horizons does a fresh layer start with?
def inv_softplus(y):
    return y + jnp.log(-jnp.expm1(-y))          # log(e^y - 1), computed stably


DT_MIN, DT_MAX = 1e-3, 1e-1

k = jax.random.PRNGKey(0)
delta0 = jnp.exp(jax.random.uniform(k, (8,), minval=math.log(DT_MIN), maxval=math.log(DT_MAX)))

print("Δ at init                :", np.round(np.asarray(delta0), 4))
print("horizon 1/(Δ|a|), a = -1 :", np.round(1.0 / np.asarray(delta0), 0), "tokens")
print("horizon 1/(Δ|a|), a = -8 :", np.round(1.0 / (8 * np.asarray(delta0)), 1), "tokens")
print("\nsoftplus(inv_softplus(Δ)) == Δ :",
      bool(jnp.allclose(jax.nn.softplus(inv_softplus(delta0)), delta0)))

## 5. The parallel scan that replaces the convolution — ≈8 min

We lost the convolution, so we need another way to compute all $L$ states without walking the
sequence one step at a time. Strip the recurrence down to its shape — everything is elementwise,
so think of $a_t$ and $b_t$ as scalars:

$$ h_t = a_t\,h_{t-1} + b_t, \qquad a_t = \bar A_t,\;\; b_t = \bar B_t x_t $$

Ask what *composing two steps* does. Apply step $i$, then step $j$:

$$ h \;\mapsto\; a_j\left(a_i h + b_i\right) + b_j \;=\; \underbrace{(a_j a_i)}_{a}\,h + \underbrace{(a_j b_i + b_j)}_{b} $$

The composition of two affine maps is an affine map. So define, on pairs,

$$ (a_i, b_i) \oplus (a_j, b_j) \;=\; \bigl(a_j a_i,\;\; a_j b_i + b_j\bigr) $$

Function composition is associative, therefore $\oplus$ is associative — and *associative is all a
prefix scan needs*. `jax.lax.associative_scan` then computes every prefix in $O(L)$ work and
$O(\log L)$ sequential depth, which is what "parallel over the sequence" means on a GPU.

Note what we did **not** need: commutativity ($\oplus$ is not commutative), linearity in $h$ of
anything nonlinear, or a fixed kernel. This is why selectivity costs us the FFT but not
parallelism — and it is the entire algorithmic content of Mamba.

In [ ]:
def scan_op(l, r):
    """Compose two affine steps. `l` is the earlier one, `r` the later one."""
    ai, bi = l
    aj, bj = r
    return (ai * aj, aj * bi + bj)


def selective_scan(a, b):
    """h_t = a_t * h_{t-1} + b_t  for all t at once, with h_{-1} = 0.

    a, b: (L, ...) -> h: (L, ...).  Everything after the leading axis is broadcast elementwise,
    so this one function handles the full (L, d_inner, d_state) state trajectory.
    """
    _, h = jax.lax.associative_scan(scan_op, (a, b))
    return h

In [ ]:
# --- check 1: is the operator actually associative? -----------------------------------------
ks = jax.random.split(jax.random.PRNGKey(1), 6)
X, Y, Z = [(jax.random.uniform(ks[2 * i], (5,)), jax.random.normal(ks[2 * i + 1], (5,)))
           for i in range(3)]
lhs, rhs = scan_op(scan_op(X, Y), Z), scan_op(X, scan_op(Y, Z))
print("(X + Y) + Z  ==  X + (Y + Z) :",
      all(bool(jnp.allclose(u, v)) for u, v in zip(lhs, rhs)))


# --- check 2: does the parallel scan agree with the definition? ------------------------------
def _sequential_reference(a, b):
    """Test oracle only: the literal definition, one step at a time. Never used by the model."""
    def step(h, ab):
        h = ab[0] * h + ab[1]
        return h, h
    return jax.lax.scan(step, jnp.zeros_like(b[0]), (a, b))[1]


ka, kb = jax.random.split(jax.random.PRNGKey(2))
a = jax.random.uniform(ka, (256, 8, 16))          # (L, d_inner, d_state), all in (0, 1)
b = jax.random.normal(kb, (256, 8, 16))
print("max |parallel - sequential| =",
      float(jnp.abs(selective_scan(a, b) - _sequential_reference(a, b)).max()))

Both checks pass, so `selective_scan` is the recurrence — just reassociated.

One honest caveat before we build on it. `associative_scan` **materialises** the whole
`(L, d_inner, d_state)` trajectory, and autodiff keeps it for the backward pass. The real Mamba
does not: its fused CUDA kernel keeps the state in SRAM, never writes it to HBM, and recomputes it
during the backward pass. Same mathematics, an order of magnitude less memory traffic. That gap is
the "other half of the contribution" the lecture kept insisting on, and it is the one part of Mamba
we are *not* reproducing here.

## 6. The block and the language model, in Equinox — ≈12 min

Two conventions, so the code reads like §4.

**Equinox modules are pytrees.** A module *is* its parameters; there is no hidden state, no
`init`/`apply` split. `eqx.filter_jit` and `eqx.filter_value_and_grad` split it into arrays (traced,
differentiated) and everything else (static) automatically.

**Every module takes one sequence, not a batch.** Shapes are `(L, d)`, exactly as in the formulas
above; the batch axis is added by `jax.vmap` at the top level. Per-token layers (`Linear`) get their
own `jax.vmap` over the length axis, which is the only place you will see a stray `vmap`.

We write RMSNorm and the causal convolution by hand — both are three lines, and the convolution's
padding is where §8's O(1) inference comes from.

In [ ]:
class RMSNorm(eqx.Module):
    scale: jax.Array
    eps: float = eqx.field(static=True)

    def __init__(self, dim, eps=1e-5):
        self.scale, self.eps = jnp.ones(dim), eps

    def __call__(self, x):                       # (..., d) -> (..., d)
        return x * jax.lax.rsqrt(jnp.mean(x * x, -1, keepdims=True) + self.eps) * self.scale


def causal_depthwise_conv(x, weight, bias):
    """x: (L, C), weight: (C, K), bias: (C,) -> (L, C).

    One independent K-tap filter per channel, no mixing across channels. Padding K-1 zeros on the
    LEFT is what makes it causal: position t sees x[t-K+1 : t+1] and nothing later.
    """
    L, K = x.shape[0], weight.shape[1]
    xp = jnp.pad(x, ((K - 1, 0), (0, 0)))
    return sum(xp[k:k + L] * weight[:, k] for k in range(K)) + bias

### The selective SSM layer

Straight transcription of the boxed formulas in §4. The `selective` flag switches between S6
(everything projected from the input) and S4-style LTI (Δ, B, C are plain parameters) — one flag,
one code path, and §9 uses it to run the controlled experiment.

In [ ]:
class SSM(eqx.Module):
    """(L, d_inner) -> (L, d_inner). Each channel carries its own N-dimensional state."""

    A_log: jax.Array                     # (d_inner, N)   A = -exp(A_log) < 0
    D: jax.Array                         # (d_inner,)     skip connection
    x_proj: eqx.nn.Linear | None         # selective: x_t -> (dt_rank | B_t | C_t)
    dt_proj: eqx.nn.Linear | None        # selective: low-rank -> Δ_t
    dt_bias: jax.Array | None            # LTI: Δ is just a parameter
    B_lti: jax.Array | None
    C_lti: jax.Array | None
    selective: bool = eqx.field(static=True)
    d_inner: int = eqx.field(static=True)
    d_state: int = eqx.field(static=True)
    dt_rank: int = eqx.field(static=True)

    def __init__(self, cfg, *, selective=True, key):
        kx, kw, kd, kb, kc = jax.random.split(key, 5)
        d_inner, N = cfg.d_inner, cfg.d_state
        self.d_inner, self.d_state, self.dt_rank = d_inner, N, cfg.dt_rank
        self.selective = selective

        # S4D-Real: a_n = -n for n = 1..N after A = -exp(A_log) — N decay rates per channel.
        self.A_log = jnp.tile(jnp.log(jnp.arange(1, N + 1, dtype=jnp.float32)), (d_inner, 1))
        self.D = jnp.ones(d_inner)

        # Δ init: log-uniform in [DT_MIN, DT_MAX], stored through the inverse of softplus so that
        # softplus(bias) reproduces it exactly at step 0.
        delta0 = jnp.exp(jax.random.uniform(kd, (d_inner,),
                                            minval=math.log(DT_MIN), maxval=math.log(DT_MAX)))

        if selective:
            self.x_proj = eqx.nn.Linear(d_inner, cfg.dt_rank + 2 * N, use_bias=False, key=kx)
            dt_proj = eqx.nn.Linear(cfg.dt_rank, d_inner, use_bias=True, key=kw)
            lim = cfg.dt_rank ** -0.5
            self.dt_proj = eqx.tree_at(
                lambda m: [m.weight, m.bias], dt_proj,
                [jax.random.uniform(kb, dt_proj.weight.shape, minval=-lim, maxval=lim),
                 inv_softplus(delta0)],
            )
            self.dt_bias = self.B_lti = self.C_lti = None
        else:
            self.x_proj = self.dt_proj = None
            self.dt_bias = inv_softplus(delta0)
            self.B_lti = jax.random.normal(kb, (N,)) * N ** -0.5
            self.C_lti = jax.random.normal(kc, (N,)) * N ** -0.5

    def _coeffs(self, u):
        """(L, d_inner) -> Δ (L, d_inner), B (L, N), C (L, N)."""
        if self.selective:
            proj = jax.vmap(self.x_proj)(u)
            dt, B, C = jnp.split(proj, [self.dt_rank, self.dt_rank + self.d_state], axis=-1)
            return jax.nn.softplus(jax.vmap(self.dt_proj)(dt)), B, C
        L = u.shape[0]
        return (jnp.broadcast_to(jax.nn.softplus(self.dt_bias), (L, self.d_inner)),
                jnp.broadcast_to(self.B_lti, (L, self.d_state)),
                jnp.broadcast_to(self.C_lti, (L, self.d_state)))

    def __call__(self, u):                                     # (L, d_inner) -> (L, d_inner)
        A = -jnp.exp(self.A_log)                               # (d_inner, N), strictly negative
        delta, B, C = self._coeffs(u)

        dA = jnp.exp(delta[:, :, None] * A[None])              # Ā_t      (L, d_inner, N)
        dBu = delta[:, :, None] * B[:, None, :] * u[:, :, None]  # B̄_t x_t (L, d_inner, N)

        h = selective_scan(dA, dBu)                            # (L, d_inner, N)
        return jnp.einsum("ldn,ln->ld", h, C) + self.D * u     # y_t = C_t·h_t + D x_t

    def step(self, u_t, h):
        """One token, O(1). u_t: (d_inner,), h: (d_inner, N) -> y_t, h_t."""
        A = -jnp.exp(self.A_log)
        if self.selective:
            dt, B, C = jnp.split(self.x_proj(u_t),
                                 [self.dt_rank, self.dt_rank + self.d_state])
            delta = jax.nn.softplus(self.dt_proj(dt))
        else:
            delta, B, C = jax.nn.softplus(self.dt_bias), self.B_lti, self.C_lti
        h = jnp.exp(delta[:, None] * A) * h + delta[:, None] * B[None] * u_t[:, None]
        return h @ C + self.D * u_t, h

### The block

`in_proj` widens `d_model` into two branches of width `d_inner`: the signal `u` that goes through
the SSM, and a gate `z` that never does. The block is
`H3 (conv + SSM) fused with a gated MLP`, which is how the paper describes it:

```
x ──▶ RMSNorm ──▶ in_proj ──┬──▶ conv1d ──▶ SiLU ──▶ SSM ──┐
                            │                              ⊙ ──▶ out_proj ──▶ (+x)
                            └──────────▶ SiLU ─────────────┘
```

Why the convolution at all, when the SSM already mixes across time? Because the SSM mixes *only*
through the state — a channel's information reaches position $t$ only if it survived the decay. A
4-tap depthwise convolution gives every channel free, exact access to its immediate neighbourhood,
which is where most of the local syntax lives. It is cheap (4 numbers per channel) and it takes
that job away from the state, leaving the state for long-range work.

In [ ]:
class MambaBlock(eqx.Module):
    norm: RMSNorm
    in_proj: eqx.nn.Linear               # d_model -> 2 * d_inner  (signal | gate)
    conv_w: jax.Array                    # (d_inner, d_conv)
    conv_b: jax.Array
    ssm: SSM
    out_proj: eqx.nn.Linear              # d_inner -> d_model

    def __init__(self, cfg, *, selective=True, key):
        ki, kc, ks, ko = jax.random.split(key, 4)
        self.norm = RMSNorm(cfg.d_model)
        self.in_proj = eqx.nn.Linear(cfg.d_model, 2 * cfg.d_inner, use_bias=False, key=ki)

        lim = cfg.d_conv ** -0.5
        self.conv_w = jax.random.uniform(kc, (cfg.d_inner, cfg.d_conv), minval=-lim, maxval=lim)
        self.conv_b = jnp.zeros(cfg.d_inner)

        self.ssm = SSM(cfg, selective=selective, key=ks)

        # n_layer branches add into one residual stream, so shrink each branch's output at init
        # (the usual GPT-2 trick) — otherwise the stream's variance grows with depth.
        out = eqx.nn.Linear(cfg.d_inner, cfg.d_model, use_bias=False, key=ko)
        self.out_proj = eqx.tree_at(lambda m: m.weight, out, out.weight / math.sqrt(cfg.n_layer))

    def __call__(self, x):                                   # (L, d_model) -> (L, d_model)
        u, z = jnp.split(jax.vmap(self.in_proj)(self.norm(x)), 2, axis=-1)
        u = jax.nn.silu(causal_depthwise_conv(u, self.conv_w, self.conv_b))
        y = self.ssm(u) * jax.nn.silu(z)                     # the gate
        return x + jax.vmap(self.out_proj)(y)

    def step(self, x_t, state):
        """One token, O(1). state = (h, buf) with buf the last d_conv-1 conv inputs."""
        h, buf = state
        u, z = jnp.split(self.in_proj(self.norm(x_t)), 2)
        win = jnp.concatenate([buf, u[None]])                # (d_conv, d_inner), oldest first
        u = jax.nn.silu((win * self.conv_w.T).sum(0) + self.conv_b)
        y, h = self.ssm.step(u, h)
        return x_t + self.out_proj(y * jax.nn.silu(z)), (h, win[1:])


class MambaLM(eqx.Module):
    tok_emb: jax.Array                   # (vocab, d_model) — tied with the output head
    blocks: list
    norm_f: RMSNorm

    def __init__(self, cfg, vocab_size, *, selective=True, key):
        ke, *kb = jax.random.split(key, cfg.n_layer + 1)
        self.tok_emb = 0.02 * jax.random.normal(ke, (vocab_size, cfg.d_model))
        self.blocks = [MambaBlock(cfg, selective=selective, key=k) for k in kb]
        self.norm_f = RMSNorm(cfg.d_model)

    def __call__(self, tokens):                              # (L,) int -> (L, vocab)
        x = self.tok_emb[tokens]
        for block in self.blocks:
            x = block(x)
        return self.norm_f(x) @ self.tok_emb.T               # weight tying

    def step(self, token, state):
        x, new = self.tok_emb[token], []
        for block, s in zip(self.blocks, state):
            x, s = block.step(x, s)
            new.append(s)
        return self.norm_f(x) @ self.tok_emb.T, new


def init_state(model, cfg):
    """Zero SSM state + zero conv buffer per layer. Zeros in the buffer == the left padding."""
    return [(jnp.zeros((cfg.d_inner, cfg.d_state)), jnp.zeros((cfg.d_conv - 1, cfg.d_inner)))
            for _ in model.blocks]

In [ ]:
model = MambaLM(cfg, vocab_size, key=jax.random.PRNGKey(cfg.seed))


def count(tree):
    return sum(x.size for x in jax.tree_util.tree_leaves(eqx.filter(tree, eqx.is_inexact_array)))


b0 = model.blocks[0]
print(f"embedding (tied)     {count(model.tok_emb) / 1e6:7.3f}M")
print(f"per block            {count(b0) / 1e6:7.3f}M")
print(f"  in_proj            {count(b0.in_proj) / 1e6:7.3f}M")
print(f"  conv               {count((b0.conv_w, b0.conv_b)) / 1e6:7.3f}M")
print(f"  ssm                {count(b0.ssm) / 1e6:7.3f}M")
print(f"  out_proj           {count(b0.out_proj) / 1e6:7.3f}M")
print(f"{'-' * 30}\ntotal                {count(model) / 1e6:7.3f}M")

out = model(xb[0])
print("\nforward:", xb[0].shape, "->", out.shape)

## 7. Training on TinyStories — ≈7 min

*Plumbing again — this is a standard language-model training loop and nothing in it is specific to
Mamba.* Worth a glance: the weight-decay mask, which excludes `A_log`. `A_log` is not a weight;
it parameterises the timescales of the state, and decaying it would silently drag every channel's
memory horizon towards the same value — the opposite of what the initialisation set up.

Start the cell, then keep reading §8 while it runs.

In [ ]:
def loss_fn(model, x, y):
    logits = jax.vmap(model)(x)                                   # (B, L, vocab)
    return optax.softmax_cross_entropy_with_integer_labels(logits, y).mean()


def make_optimizer(cfg, model):
    schedule = optax.warmup_cosine_decay_schedule(
        init_value=0.0, peak_value=cfg.lr, warmup_steps=cfg.warmup,
        decay_steps=cfg.steps, end_value=cfg.lr_min,
    )
    params = eqx.filter(model, eqx.is_inexact_array)
    # decay matrices, not vectors — and never A_log, which is a timescale, not a weight
    mask = jax.tree_util.tree_map(lambda p: p.ndim >= 2, params)
    mask = eqx.tree_at(lambda m: [b.ssm.A_log for b in m.blocks], mask,
                       replace=[False] * cfg.n_layer)
    return optax.chain(
        optax.clip_by_global_norm(cfg.grad_clip),
        optax.adamw(schedule, b1=0.9, b2=0.95, weight_decay=cfg.weight_decay, mask=mask),
    )


model = MambaLM(cfg, vocab_size, key=jax.random.PRNGKey(cfg.seed))   # fresh, so this is re-runnable
optim = make_optimizer(cfg, model)
opt_state = optim.init(eqx.filter(model, eqx.is_inexact_array))


@eqx.filter_jit
def train_step(model, opt_state, x, y):
    loss, grads = eqx.filter_value_and_grad(loss_fn)(model, x, y)
    updates, opt_state = optim.update(grads, opt_state, eqx.filter(model, eqx.is_inexact_array))
    return eqx.apply_updates(model, updates), opt_state, loss, optax.global_norm(grads)


@eqx.filter_jit
def eval_step(model, x, y):
    return loss_fn(model, x, y)


@eqx.filter_jit
def sample_next(model, ctx, key, temperature=0.8, top_k=40):
    logits = model(ctx)[-1] / temperature
    kth = jax.lax.top_k(logits, top_k)[0][-1]
    return jax.random.categorical(key, jnp.where(logits < kth, -jnp.inf, logits))


def generate_windowed(model, key, prompt, n_new=120, window=128, **kw):
    """The obvious way to sample: re-run the whole prefix for every new token.

    The window is fixed so that jit compiles once. Cost per token is O(window) — §8 fixes that.
    """
    ids = [eot] * window + [int(t) for t in prompt]
    for _ in range(n_new):
        key, sk = jax.random.split(key)
        ids.append(int(sample_next(model, jnp.asarray(ids[-window:], dtype=jnp.int32), sk, **kw)))
    return ids[window + len(prompt):]                        # only what was generated

In [ ]:
rng = np.random.default_rng(cfg.seed)
key = jax.random.PRNGKey(cfg.seed + 1)
prompt = encode("Once upon a time")

hist = {"step": [], "train": [], "val": [], "gnorm": []}
t_start, tokens_seen = time.perf_counter(), 0

for step in range(1, cfg.steps + 1):
    xb, yb = get_batch(rng, "train")
    model, opt_state, loss, gnorm = train_step(model, opt_state, xb, yb)
    tokens_seen += xb.size

    if step % cfg.eval_every == 0 or step == 1:
        val = np.mean([float(eval_step(model, *get_batch(rng, "val")))
                       for _ in range(cfg.eval_batches)])
        hist["step"].append(step)
        hist["train"].append(float(loss))
        hist["val"].append(float(val))
        hist["gnorm"].append(float(gnorm))

        key, gk = jax.random.split(key)
        sample = decode(generate_windowed(model, gk, prompt, n_new=80))

        clear_output(wait=True)
        fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
        ax[0].plot(hist["step"], hist["train"], "-o", ms=3, label="train")
        ax[0].plot(hist["step"], hist["val"], "-o", ms=3, label="val")
        ax[0].set_xlabel("step"); ax[0].set_ylabel("cross-entropy"); ax[0].legend(fontsize=8)
        ax[0].set_title(f"loss  (val {val:.3f}  |  ppl {math.exp(val):.1f})")
        ax[1].plot(hist["step"], hist["gnorm"], "-o", ms=3, c="C3")
        ax[1].set_xlabel("step"); ax[1].set_title("gradient norm (pre-clip)")
        plt.tight_layout(); plt.show()

        el = time.perf_counter() - t_start
        print(f"step {step:>5}/{cfg.steps}   train {float(loss):.3f}   val {val:.3f}   "
              f"|g| {float(gnorm):.2f}   {tokens_seen / el / 1e3:.1f}k tok/s   {el:.0f}s elapsed")
        print(f"\n  Once upon a time{sample}")
    else:
        print(f"\rstep {step}/{cfg.steps}", end="")

Two things to look at in the sample, rather than the loss number.

**Where it gets English first.** Word shapes and local grammar come almost immediately — that is
the convolution and the fast-decaying channels. Agreement across a clause takes longer, and that
is the state doing its job.

**Where it fails.** Watch for a name introduced in the first sentence and then quietly replaced by
a different one later. That is the fixed-size state losing a specific fact — exactly the weakness
the lecture named (associative recall, copying, needle-in-a-haystack) and the reason production
models are hybrids rather than pure SSMs.

## 8. O(1) inference: the same model as an RNN — ≈5 min

Everything so far ran the parallel form. But §2 gave us a recurrence, so at generation time we can
just *be* an RNN: carry the state forward and never look at the prefix again.

What has to be carried, per layer:

- the SSM state `h`, shape `(d_inner, d_state)` — that is the whole point;
- the convolution's last `d_conv - 1` inputs, shape `(d_conv - 1, d_inner)` — because a 4-tap
  filter needs three previous values. Initialising this buffer to **zeros** is exactly the left
  padding from `causal_depthwise_conv`, which is why the two forms agree from the very first token.

Nothing else. No cache that grows with the sequence — that is the property the whole architecture
was built to have.

`MambaBlock.step` and `MambaLM.step` in §6 already implement this. The only honest way to know they
are right is to check them against the parallel path.

In [ ]:
@eqx.filter_jit
def rnn_step(model, token, state):
    return model.step(token, state)


probe = jnp.asarray(val_data[:64].astype(np.int32))

parallel = model(probe)                                   # (64, vocab), all positions at once
state, rows = init_state(model, cfg), []
for t in probe:                                           # one token at a time, O(1) each
    logits, state = rnn_step(model, t, state)
    rows.append(logits)
recurrent = jnp.stack(rows)

err = float(jnp.abs(parallel - recurrent).max())
print(f"max |parallel - recurrent| = {err:.2e}   (float32 accumulation noise, not a bug)")
assert err < 1e-2, "the two forms disagree — the conv buffer or the state is wired wrong"

Same model, same numbers, two execution modes. Now the part that motivated all of it: cost per
token as the context grows.

In [ ]:
def _bench(fn, n, warmup=3):
    for _ in range(warmup):
        fn()
    jax.block_until_ready(fn())
    t0 = time.perf_counter()
    for _ in range(n):
        out = fn()
    jax.block_until_ready(out)
    return n / (time.perf_counter() - t0)


@eqx.filter_jit
def _last_logits(model, ctx):
    return model(ctx)[-1]


state0 = init_state(model, cfg)
tok0 = jnp.asarray(0, dtype=jnp.int32)
rec = _bench(lambda: rnn_step(model, tok0, state0), n=100)

print(f"{'context':>9} | {'re-run the prefix':>18} | {'recurrent step':>15}")
print("-" * 50)
for window in (128, 512, 2048):
    ctx = jnp.zeros((window,), dtype=jnp.int32)
    naive = _bench(lambda: _last_logits(model, ctx), n=20)
    print(f"{window:>9} | {naive:>13.1f} tok/s | {rec:>10.1f} tok/s")

The left column degrades with context; the right one is a flat line, by construction. (At this
model size a large share of the recurrent number is Python dispatch overhead rather than compute —
one `jax.lax.scan` over the whole generation would remove it. The *shape* of the comparison is the
point, not the absolute numbers.)

In [ ]:
def generate(model, key, prompt, n_new=200, temperature=0.8, top_k=40):
    """Constant memory, constant time per token."""
    state, out = init_state(model, cfg), []
    for t in prompt[:-1]:                                    # absorb the prompt
        _, state = rnn_step(model, jnp.asarray(t, dtype=jnp.int32), state)

    nxt = jnp.asarray(prompt[-1], dtype=jnp.int32)
    for _ in range(n_new):
        logits, state = rnn_step(model, nxt, state)
        key, sk = jax.random.split(key)
        logits = logits / temperature
        kth = jax.lax.top_k(logits, top_k)[0][-1]
        nxt = jax.random.categorical(sk, jnp.where(logits < kth, -jnp.inf, logits))
        out.append(int(nxt))
    return out


key, gk = jax.random.split(key)
print("Once upon a time" + decode(generate(model, gk, prompt, n_new=250)))

## 9. Playground — talk to the model

The model is trained, and §8 gave us a generator whose cost per token does not depend on how long
the context is. So: type something and watch what a state space model does with it.

Every cell below runs the **recurrent** path from §8 — a fixed-size state plus a three-element
convolution buffer, with no copy of the prefix kept anywhere. Whatever comes out of these cells
came out of `d_inner × d_state` numbers per layer, and nothing else.

In [ ]:
def play(prompt, n_new=200, temperature=0.8, top_k=40, seed=None, width=96):
    """Continue `prompt`. Re-run the cell for a new sample; pass seed=... to pin one down."""
    global _play_key
    if seed is None:
        _play_key, k = jax.random.split(_play_key)
    else:
        k = jax.random.PRNGKey(seed)

    ids = encode(prompt, strict=False) or [eot]
    text = prompt + decode(generate(model, k, ids, n_new=n_new,
                                    temperature=temperature, top_k=top_k))
    print(textwrap.fill(text, width), "\n")
    return text


_play_key = jax.random.PRNGKey(1234)

play("Once upon a time, there was a little girl named Sara. She")

### Change the prompt and re-run

This is the cell to edit during the seminar. Some prompts worth trying, and what each one probes:

| prompt | what you are testing |
|---|---|
| `"One day, Tom found a shiny"` | ordinary in-distribution continuation |
| `"The dog was sad because"` | can it carry a causal clause across the comma? |
| `"Suddenly, the door opened and"` | does it commit to a new character, or drift? |
| `"import numpy as np"` | out-of-distribution — watch it snap back to children's stories |
| `"Лена пошла гулять"` | a prompt the pruned vocabulary cannot fully represent |

In [ ]:
PROMPT = "One day, Tom found a shiny"

play(PROMPT, n_new=150)
play(PROMPT, n_new=150)
play(PROMPT, n_new=150)

Three samples, one prompt. The model is a distribution, not a function; the only thing that differs
between those runs is the random draw taken at each step.

### What the two sampling knobs do

`temperature` divides the logits before sampling and `top_k` truncates to the `k` most likely
tokens. Low temperature concentrates the distribution and the model repeats its safest
continuation; high temperature flattens it, and grammar is the first thing to go.

All four runs below share one seed on purpose, so the random draws are the *same* at every step and
the only thing varying is the sharpness of the distribution they are drawn from. A consequence
worth predicting before you run it: two temperatures that never change which token wins will
produce byte-identical text. Drop the `seed=` argument to get independent samples instead.

In [ ]:
for t in (0.2, 0.7, 1.0, 1.4):
    print(f"--- temperature {t}")
    play(PROMPT, n_new=80, temperature=t, seed=7)

### The fixed state, made visible

§7 asked you to watch for a name introduced early and quietly replaced later. Here is that test run
on purpose: introduce a character, generate far past the point where a fixed-size summary has had
to start discarding things, and count who actually shows up.

Whatever you see here is the architecture's defining limitation rather than a training bug. The
state holds `d_inner × d_state` numbers and not one more; a Transformer at this point would still
have the literal token sitting in its KV cache. This is the retrieval gap that makes production
models hybrids instead of pure SSMs.

In [ ]:
NAME_PROMPT = "Once upon a time there was a girl named Lila. Lila had a red ball."

story = play(NAME_PROMPT, n_new=350, temperature=0.7, seed=3)

tail = story[len(NAME_PROMPT):]
stop = {"The", "She", "He", "They", "It", "But", "And", "Then", "One", "Her", "His"}
names = [w for w in re.findall(r"\b[A-Z][a-z]+\b", tail) if w not in stop]
print("characters named after the prompt:", collections.Counter(names).most_common(8))
print("does 'Lila' survive to the end?", "yes" if "Lila" in tail[-200:] else "no")

### What the model decided to skip

Here is the one quantity in a selective SSM with no analogue in a Transformer: $\Delta_t$, the
per-token step size from §2. Small $\Delta_t$ means $\bar A \to I$ and $\bar B \to 0$ — *this token
did not move the state*. Large $\Delta_t$ means the state was refreshed. It is the model's own
answer to "was this token worth anything", and it is a plain array we can read straight out.

We recover it by replaying the front half of one block. Nothing here is special-purpose: it calls
the same `_coeffs` the forward pass calls.

In [ ]:
def layer_deltas(model, tokens, layer=0):
    """Δ_t for every channel of one layer: (L, d_inner)."""
    x = model.tok_emb[jnp.asarray(tokens, dtype=jnp.int32)]
    for block in model.blocks[:layer]:
        x = block(x)
    block = model.blocks[layer]
    u, _ = jnp.split(jax.vmap(block.in_proj)(block.norm(x)), 2, axis=-1)
    u = jax.nn.silu(causal_depthwise_conv(u, block.conv_w, block.conv_b))
    delta, _, _ = block.ssm._coeffs(u)
    return np.asarray(delta)


PROBE = "Once upon a time, there was a little girl named Sara. She had a big red balloon."
LAYER = cfg.n_layer // 2

ids = encode(PROBE, strict=False)
pieces = [decode([i]) for i in ids]
d = layer_deltas(model, ids, layer=LAYER).mean(-1)          # average over the d_inner channels

order = np.argsort(d)
print(f"layer {LAYER}, Δ averaged over {cfg.d_inner} channels\n")
print("left the state alone (smallest Δ):")
print("   ", "  ".join(repr(pieces[i]) for i in order[:8]))
print("refreshed the state (largest Δ):")
print("   ", "  ".join(repr(pieces[i]) for i in order[::-1][:8]))

plt.figure(figsize=(11, 3))
plt.plot(d, "-o", ms=3)
plt.xticks(range(len(pieces)), [p.strip() or "␣" for p in pieces], rotation=90, fontsize=7)
plt.ylabel(r"mean $\Delta_t$")
plt.title(f"how much each token moved the state  (layer {LAYER})")
plt.tight_layout()
plt.show()

One caveat, said out loud: after ten minutes of training on 5M tokens this plot is suggestive, not
conclusive, and individual spikes are not worth a story. What it *is* good for is the shape of the
thing. A time-invariant S4 layer would give you a perfectly horizontal line here, by construction —
its $\Delta$ is a parameter and cannot depend on the token at all. That this line moves is
selectivity, and that is the whole difference between §3 and §4.

Things worth pulling on, if you have time left:

- run `layer_deltas` on the first and the last layer and compare — do they skip the same tokens?
- set `top_k=1` to make sampling greedy, and see how many tokens it takes to fall into a loop;
- feed a sentence, then the same sentence again, and check whether $\Delta_t$ differs on the second
  copy — that would be the model recognising something it has already stored.